# AlphaTrend Backtest — 5H Bars, Portfolio, Long-Only

Faithful Python port of KivancOzbilgic's **AlphaTrend** Pine v5 indicator, run as a long-only portfolio strategy.

**Rules**
- **Timeframe:** 5H bars (built by grouping 5 consecutive 1H bars).
- **History:** **last 12 months** (`HISTORY_D = 365`).
- **Entry:** BUY signal (`crossover(AlphaTrend, AlphaTrend[2])`) → fill at **next bar's open**.
- **Exit:** SELL signal (`crossunder(AlphaTrend, AlphaTrend[2])`) → fill at next bar's open.
- **No re-entry** on the same ticker until a new BUY signal.
- **Portfolio:** single shared cash balance, starts at **$10,000**.
- **Concurrency cap:** at most **6 open positions** at once.
- **Leverage cap:** total notional across open positions ≤ **1.8 × current balance**.
- **Sizing:** each new entry = `balance × 0.30` notional (= 1.8 / 6), clipped if the leverage cap would be breached.
- **Compounding:** balance updates after every closed trade (win or loss); the next trade is sized off the remaining balance.

**Run in Google Colab:** *File → Upload notebook* → pick this `.ipynb` → **Runtime → Run all**.

Edit the **Configuration** cell to change tickers or risk caps.

In [ ]:
# --- Colab / local setup ---
import sys, subprocess
for pkg in ('yfinance', 'plotly'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

import plotly.io as pio
try:
    import google.colab  # noqa: F401
    pio.renderers.default = 'colab'
    IN_COLAB = True
except ImportError:
    pio.renderers.default = 'notebook_connected'
    IN_COLAB = False
print('Environment:', 'Google Colab' if IN_COLAB else 'local Jupyter')

## ⚙️ Configuration — edit this cell

Add or remove tickers in `TICKERS`. Examples of valid yfinance symbols:
- US stocks: `'AAPL'`, `'MSFT'`, `'NVDA'`
- Crypto:    `'BTC-USD'`, `'ETH-USD'`
- Forex:     `'EURUSD=X'`
- Indexes:   `'^GSPC'` (S&P 500)
- Turkish:   `'THYAO.IS'`, `'ASELS.IS'`

Note: yfinance only provides ~730 days of intraday (1H) history, which caps the 5H window.

In [ ]:
# ============================================================
#  >>> ADD YOUR TICKERS HERE <<<
# ============================================================
TICKERS = [
    'AAPL',
    'MSFT',
    'NVDA',
    'TSLA',
    'AMD',
    'META',
    'GOOGL',
    'BTC-USD',
]

# --- Data window: last 12 months ---
HISTORY_D     = 365      # backtest the last 12 months
BARS_PER_5H   = 5        # group 5 hourly bars into one 5H bar — DO NOT change

# --- AlphaTrend indicator params (match Pine inputs) ---
COEFF         = 1.0      # Multiplier
AP            = 14       # Common Period
USE_MFI       = True     # True = MFI (Pine default). False = RSI mode (`novolumedata=true`).
                         # Use False for FX / indexes that lack reliable volume.

# --- Portfolio risk ---
START_BAL     = 10_000.0 # starting cash
MAX_LEVERAGE  = 1.8      # total notional across open positions <= MAX_LEVERAGE * balance
MAX_POSITIONS = 6        # at most 6 concurrent open positions
# Per-entry sizing: balance * (MAX_LEVERAGE / MAX_POSITIONS) = balance * 0.30
# clipped down so total notional never exceeds MAX_LEVERAGE * balance.

# --- Costs ---
COMM_RATE     = 0.0015   # 0.15% per side, applied to traded notional
SLIPPAGE      = 0.0      # additional cost fraction per side (e.g. 0.0005 = 5 bps)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go

In [ ]:
# --- Data fetch: 1H bars → group into 5H bars ---
def fetch_1h(ticker, days=HISTORY_D):
    end = pd.Timestamp.utcnow().tz_localize(None)
    start = end - pd.Timedelta(days=days)
    df = yf.download(ticker, start=start, end=end, interval='1h',
                     auto_adjust=True, progress=False, threads=False)
    if df.empty:
        return df
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()

def resample_to_5h(df_1h, n=BARS_PER_5H):
    if df_1h.empty:
        return df_1h
    grp = np.arange(len(df_1h)) // n
    out = pd.DataFrame({
        'Open':   df_1h['Open'].groupby(grp).first(),
        'High':   df_1h['High'].groupby(grp).max(),
        'Low':    df_1h['Low'].groupby(grp).min(),
        'Close':  df_1h['Close'].groupby(grp).last(),
        'Volume': df_1h['Volume'].groupby(grp).sum(),
    })
    out.index = df_1h.index.to_series().groupby(grp).first().values
    return out

In [ ]:
# --- AlphaTrend indicator (faithful Pine v5 port) ---
def true_range(df):
    prev_close = df['Close'].shift(1)
    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - prev_close).abs(),
        (df['Low']  - prev_close).abs(),
    ], axis=1).max(axis=1)
    tr.iloc[0] = np.nan
    return tr

def mfi(df, period):
    tp = (df['High'] + df['Low'] + df['Close']) / 3.0
    raw_mf = tp * df['Volume']
    delta = tp.diff()
    pos_mf = raw_mf.where(delta > 0, 0.0)
    neg_mf = raw_mf.where(delta < 0, 0.0)
    mfr = pos_mf.rolling(period).sum() / neg_mf.rolling(period).sum().replace(0, np.nan)
    return 100.0 - (100.0 / (1.0 + mfr))

def rsi(series, period):
    delta = series.diff()
    up = delta.clip(lower=0)
    dn = (-delta).clip(lower=0)
    avg_up = up.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    avg_dn = dn.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    rs = avg_up / avg_dn.replace(0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))

def alphatrend(df, coeff=COEFF, ap=AP, use_mfi=USE_MFI):
    tr  = true_range(df)
    atr = tr.rolling(ap).mean()
    up_t   = df['Low']  - atr * coeff
    down_t = df['High'] + atr * coeff
    osc = mfi(df, ap) if use_mfi else rsi(df['Close'], ap)

    out  = np.full(len(df), np.nan)
    prev = 0.0  # nz(AlphaTrend[1]) defaults to 0
    for i in range(len(df)):
        ut, dt = up_t.iloc[i], down_t.iloc[i]
        if np.isnan(ut) or np.isnan(dt):
            continue
        o = osc.iloc[i]
        bullish = (not np.isnan(o)) and (o >= 50)
        val = (prev if ut < prev else ut) if bullish else (prev if dt > prev else dt)
        out[i] = val
        prev = val
    return pd.Series(out, index=df.index, name='AT')

def compute_signals(df):
    at = alphatrend(df)
    df = df.copy()
    df['AT'] = at
    a0, a1, a2, a3 = at, at.shift(1), at.shift(2), at.shift(3)
    df['BuySig']  = ((a0 > a2) & (a1 <= a3)).fillna(False)
    df['SellSig'] = ((a0 < a2) & (a1 >= a3)).fillna(False)
    return df

In [ ]:
# --- Build time-sorted event list across all tickers ---
def build_events(per_ticker):
    """Signal on bar i → execution at bar i+1 open. At identical timestamps,
    process SELLs first to free capacity; then BUYs in alphabetical order."""
    events = []
    for ticker, df in per_ticker.items():
        opens = df['Open'].values
        idx   = df.index
        bs    = df['BuySig'].values
        ss    = df['SellSig'].values
        for i in range(len(df) - 1):
            px = float(opens[i + 1])
            if np.isnan(px):
                continue
            if bs[i]:
                events.append((idx[i + 1], 'BUY',  ticker, px))
            if ss[i]:
                events.append((idx[i + 1], 'SELL', ticker, px))
    events.sort(key=lambda e: (e[0], 0 if e[1] == 'SELL' else 1, e[2]))
    return events

In [ ]:
# --- Portfolio simulator: up to MAX_POSITIONS concurrent, total <= MAX_LEVERAGE * balance ---
#
# Compounding model:
#   `balance` is the live cash balance. It starts at START_BAL and is updated
#   after EVERY closed trade by `balance += net_pnl` (win or loss). The next
#   BUY signal then sizes off this updated balance, so each subsequent trade
#   automatically uses the remaining capital with the same leverage rules.
#
# Per-entry sizing:
#   notional = min(balance * (MAX_LEVERAGE / MAX_POSITIONS), headroom)
#   where headroom = MAX_LEVERAGE * balance - (sum of currently-open notionals)
#   With defaults (1.8 / 6) each new position gets 30% of the current balance.
#   When 6 are open, total notional = 1.8 * balance (the cap).
def portfolio_simulate(events, per_ticker,
                       start_balance=START_BAL,
                       max_leverage=MAX_LEVERAGE,
                       max_positions=MAX_POSITIONS,
                       comm=COMM_RATE, slippage=SLIPPAGE):
    balance      = start_balance
    open_pos     = {}    # ticker -> position dict
    trades       = []    # closed trades
    skipped      = []    # BUY signals dropped because no capacity
    per_pos_frac = max_leverage / max_positions   # 0.30 with defaults

    first_time = events[0][0] if events else None
    equity_curve = [(first_time, balance, len(open_pos))]

    def total_notional():
        return sum(p['notional'] for p in open_pos.values())

    for time, kind, ticker, price in events:
        if kind == 'SELL':
            if ticker not in open_pos:
                continue
            pos        = open_pos.pop(ticker)
            eff_exit   = price * (1.0 - slippage)
            ret        = eff_exit / pos['entry_price'] - 1.0
            gross_pnl  = ret * pos['notional']
            exit_comm  = comm * pos['notional'] * (eff_exit / pos['entry_price'])
            total_comm = pos['entry_comm'] + exit_comm
            net_pnl    = gross_pnl - total_comm
            # >>> Compounding: realized P&L (win OR loss) updates the balance
            #     that the NEXT BUY will be sized against. <<<
            balance   += net_pnl

            trades.append({
                'ticker':           ticker,
                'entry_time':       pos['entry_time'],
                'entry_price':      pos['entry_price'],
                'exit_time':        time,
                'exit_price':       eff_exit,
                'notional':         pos['notional'],
                'balance_at_entry': pos['balance_at_entry'],
                'return_pct':       ret * 100.0,
                'gross_pnl':        gross_pnl,
                'commission':       total_comm,
                'net_pnl':          net_pnl,
                'balance_after':    balance,
            })
            equity_curve.append((time, balance, len(open_pos)))

        elif kind == 'BUY':
            if ticker in open_pos:
                continue  # already in position on this ticker
            if balance <= 0:
                skipped.append((time, ticker, 'balance_depleted'))
                continue
            if len(open_pos) >= max_positions:
                skipped.append((time, ticker, 'max_positions'))
                continue
            # Size from the CURRENT balance -- this is what makes the strategy
            # compound: after a win, next notional is bigger; after a loss, smaller.
            headroom = balance * max_leverage - total_notional()
            full     = balance * per_pos_frac
            notional = min(full, headroom)
            if notional <= 0:
                skipped.append((time, ticker, 'leverage_cap'))
                continue

            eff_entry  = price * (1.0 + slippage)
            entry_comm = comm * notional
            open_pos[ticker] = {
                'entry_time':       time,
                'entry_price':      eff_entry,
                'notional':         notional,
                'entry_comm':       entry_comm,
                'balance_at_entry': balance,
            }
            equity_curve.append((time, balance, len(open_pos)))

    # Still-open trades at end of data
    open_trades = []
    for ticker, pos in open_pos.items():
        last_close = float(per_ticker[ticker]['Close'].iloc[-1])
        ret        = last_close / pos['entry_price'] - 1.0
        exit_comm  = comm * pos['notional'] * (last_close / pos['entry_price'])
        unreal_net = ret * pos['notional'] - pos['entry_comm'] - exit_comm
        open_trades.append({
            'ticker':             ticker,
            **pos,
            'last_price':         last_close,
            'unrealized_pct':     ret * 100.0,
            'unrealized_net_pnl': unreal_net,
        })
    projected_balance = balance + sum(t['unrealized_net_pnl'] for t in open_trades)
    return trades, skipped, equity_curve, open_trades, projected_balance


In [ ]:
# --- Fetch data & compute signals for every ticker ---
per_ticker = {}
for t in TICKERS:
    print(f'Fetching {t}...', end=' ')
    raw = fetch_1h(t)
    if raw.empty:
        print('NO DATA, skipping')
        continue
    bars = resample_to_5h(raw)
    if len(bars) < AP + 5:
        print(f'only {len(bars)} 5H bars, skipping')
        continue
    sig = compute_signals(bars)
    per_ticker[t] = sig
    print(f'{len(bars)} 5H bars  |  BUY={int(sig.BuySig.sum())}  SELL={int(sig.SellSig.sum())}')

events = build_events(per_ticker)
print(f'\nTotal raw signal events: {len(events)}')

In [ ]:
# --- Run portfolio simulation ---
trades, skipped, equity_curve, open_trades, projected_bal = portfolio_simulate(events, per_ticker)
trades_df = pd.DataFrame(trades)

final_bal = equity_curve[-1][1]
realized  = (final_bal / START_BAL - 1.0) * 100.0
projected = (projected_bal / START_BAL - 1.0) * 100.0

print(f'Starting balance:  ${START_BAL:,.2f}')
print(f'Closed trades:     {len(trades_df)}')
print(f'Skipped BUYs:      {len(skipped)}  (no capacity at signal time)')
print(f'Open positions:    {len(open_trades)} of max {MAX_POSITIONS}')
print(f'Final cash:        ${final_bal:,.2f}  ({realized:+.2f}% realized)')
if open_trades:
    print(f'Projected (incl unrealized): ${projected_bal:,.2f}  ({projected:+.2f}%)')

if skipped:
    reasons = pd.Series([r for _, _, r in skipped]).value_counts()
    print('\nSkip reasons:')
    print(reasons.to_string())

In [ ]:
# --- Compounding verification: first 10 trades chronologically ---
# Shows that each successive trade is sized off the balance left over
# from the previous trade(s). Notional_at_entry should equal
# round(balance_at_entry * (MAX_LEVERAGE / MAX_POSITIONS), 2)
# unless the leverage cap clipped it.
if trades_df.empty:
    print('No closed trades to verify.')
else:
    expected_frac = MAX_LEVERAGE / MAX_POSITIONS
    chk = trades_df[['ticker','entry_time','balance_at_entry','notional','balance_after','net_pnl']].head(10).copy()
    chk['expected_notional'] = (chk['balance_at_entry'] * expected_frac).round(2)
    chk['entry_time'] = pd.to_datetime(chk['entry_time']).dt.strftime('%Y-%m-%d %H:%M')
    for c in ('balance_at_entry','notional','balance_after','net_pnl','expected_notional'):
        chk[c] = chk[c].round(2)
    print('First 10 trades — compounding check')
    print('  notional should ~= balance_at_entry * %.2f  (cap may clip when many positions open)' % expected_frac)
    display(chk)

## Trade log (all closed trades, chronological)

In [ ]:
if trades_df.empty:
    print('No closed trades.')
else:
    cols = ['ticker','entry_time','entry_price','exit_time','exit_price',
            'notional','balance_at_entry','return_pct',
            'gross_pnl','commission','net_pnl','balance_after']
    disp = trades_df[cols].copy()
    disp['entry_time'] = pd.to_datetime(disp['entry_time']).dt.strftime('%Y-%m-%d %H:%M')
    disp['exit_time']  = pd.to_datetime(disp['exit_time']).dt.strftime('%Y-%m-%d %H:%M')
    for c in ('entry_price','exit_price','notional','balance_at_entry',
              'return_pct','gross_pnl','commission','net_pnl','balance_after'):
        disp[c] = disp[c].round(2)
    display(disp)

## Per-ticker summary

In [ ]:
def summarize(group):
    n = len(group)
    wins   = group[group['net_pnl'] > 0]
    losses = group[group['net_pnl'] <= 0]
    return pd.Series({
        'trades':       n,
        'wins':         len(wins),
        'losses':       len(losses),
        'win_rate_%':   round(100.0 * len(wins) / n, 1) if n else 0.0,
        'avg_win_$':    round(wins['net_pnl'].mean(), 2)   if len(wins) else 0.0,
        'avg_loss_$':   round(losses['net_pnl'].mean(), 2) if len(losses) else 0.0,
        'best_$':       round(group['net_pnl'].max(), 2)   if n else 0.0,
        'worst_$':      round(group['net_pnl'].min(), 2)   if n else 0.0,
        'total_net_$':  round(group['net_pnl'].sum(), 2)   if n else 0.0,
        'total_comm_$': round(group['commission'].sum(), 2) if n else 0.0,
    })

if trades_df.empty:
    print('No closed trades.')
else:
    by_ticker = trades_df.groupby('ticker').apply(summarize)
    overall   = pd.DataFrame({'ALL': summarize(trades_df)}).T
    display(pd.concat([by_ticker, overall]))

    portfolio = pd.DataFrame([{
        'starting_balance_$':  round(START_BAL, 2),
        'final_cash_$':        round(final_bal, 2),
        'realized_return_%':   round(realized, 2),
        'projected_balance_$': round(projected_bal, 2),
        'projected_return_%':  round(projected, 2),
        'closed_trades':       len(trades_df),
        'skipped_buys':        len(skipped),
        'max_leverage':        MAX_LEVERAGE,
        'max_positions':       MAX_POSITIONS,
        'open_positions':      len(open_trades),
    }])
    print('\nPortfolio-level summary:')
    display(portfolio)

## Open positions (still in trade at end of data)

In [ ]:
if open_trades:
    op_rows = []
    for ot in open_trades:
        op_rows.append({
            'ticker':           ot['ticker'],
            'entry_time':       pd.to_datetime(ot['entry_time']).strftime('%Y-%m-%d %H:%M'),
            'entry_price':      round(ot['entry_price'], 2),
            'last_price':       round(ot['last_price'], 2),
            'notional':         round(ot['notional'], 2),
            'balance_at_entry': round(ot['balance_at_entry'], 2),
            'unrealized_%':     round(ot['unrealized_pct'], 2),
            'unrealized_$':     round(ot['unrealized_net_pnl'], 2),
        })
    display(pd.DataFrame(op_rows).set_index('ticker'))
else:
    print('No open positions at end of data.')

## P&L chart — portfolio balance over time

In [ ]:
if not equity_curve or len(equity_curve) <= 1:
    print('No closed trades — balance unchanged from starting value.')
else:
    times = [t for t, _, _ in equity_curve]
    bals  = [b for _, b, _ in equity_curve]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=times, y=bals, mode='lines+markers',
        name='Cash balance', line=dict(color='#00874c', width=2),
        hovertemplate='%{x|%Y-%m-%d %H:%M}<br>$%{y:,.2f}<extra></extra>',
    ))
    fig.add_hline(
        y=START_BAL, line=dict(color='gray', dash='dash'),
        annotation_text=f'Start ${START_BAL:,.0f}', annotation_position='top left',
    )
    if not trades_df.empty:
        m = trades_df[['exit_time','ticker','balance_after']].copy()
        fig.add_trace(go.Scatter(
            x=m['exit_time'], y=m['balance_after'], mode='markers', name='Trade close',
            text=m['ticker'], hovertemplate='%{text} → $%{y:,.2f}<extra></extra>',
            marker=dict(symbol='circle', size=8, color='#0022FC',
                        line=dict(color='white', width=1)),
        ))
    fig.update_layout(
        title=f'Portfolio P&L — start ${START_BAL:,.0f} → end ${bals[-1]:,.2f} ({realized:+.2f}%)',
        height=460, margin=dict(l=40, r=20, t=60, b=30),
        xaxis_title='Time', yaxis_title='Balance ($)',
    )
    fig.show()